# Full-size training and experiments on Colab

Runs the GPU part of the project: unconditional and conditional EGNN training, sampling with
every conditioning method, relaxation, evaluation and the summary table.

**Before you start**
1. *Runtime → Change runtime type → GPU.* A T4 (free tier) works; an L4 or A100 is faster.
2. Runs are written to **Google Drive** (`MyDrive/idl-project/`), so nothing is lost when Colab
   disconnects. Training resumes from its last checkpoint and finished experiments are skipped,
   so after a disconnect just run the cells again from the top.
3. The datasets are already in the repository (`data/`); nothing has to be generated.

**Order:** setup → GPU check → speed test → unconditional training → conditional training →
experiments → summary. Each training cell runs for at most `MAX_MINUTES` per session.

In [ ]:
# 0. Is there a GPU?
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 1. Google Drive for runs and results (persists across sessions)
from google.colab import drive
drive.mount("/content/drive")
import os
WORK = "/content/drive/MyDrive/idl-project"
os.makedirs(f"{WORK}/runs", exist_ok=True)
os.makedirs(f"{WORK}/eval", exist_ok=True)
os.makedirs(f"{WORK}/results", exist_ok=True)
print("runs and results go to", WORK)

In [ ]:
# 2. Code: clone (or update) the repository and install the package
%cd /content
!test -d IDL-PROJECT || git clone -q --depth 1 https://github.com/chilalaabubakar/IDL-PROJECT.git
%cd /content/IDL-PROJECT
!git pull -q
!pip install -q -e ".[dev]"
!git log --oneline -1

In [ ]:
# 3. Quick check (~1-2 min): symmetry tests, the sampler, and the whole pipeline end to end.
# train/sample/relax pick the GPU automatically, so this also exercises the CUDA code paths.
!python -m pytest -q tests/test_equivariance.py tests/test_sampler.py tests/test_pipeline.py

## Speed test

200 updates of the full-size model (`configs/train/uncond.yaml`: hidden 128, 4 layers,
batch 16) to estimate how long the 40,000-update run takes on this GPU.

In [ ]:
!python scripts/train.py --config configs/train/uncond.yaml name=speedtest runs_root=/content/speedtest optim.n_updates=200 val_every=200 log_every=50
import csv, glob
log = sorted(glob.glob("/content/speedtest/*_speedtest/log.csv"))[-1]
row = list(csv.DictReader(open(log)))[-1]
sec_per_step = float(row["seconds"]) / float(row["step"])
print(f"{sec_per_step:.3f} s/update -> 40,000 updates ~ {40000 * sec_per_step / 3600:.1f} h "
      f"(conditional, 50,000 updates ~ {50000 * sec_per_step / 3600:.1f} h)")

## Training

`train()` starts a run in Drive, or **resumes** the latest run of the same name from its
checkpoint. Each call trains for at most `MAX_MINUTES`; run the cell again (after a
disconnect, or to continue) until it prints that the run is finished. Checkpoints are saved
every 1,000 updates.

In [ ]:
import glob, subprocess, sys

MAX_MINUTES = 240  # per session; keep below your Colab session limit


def latest_run(name):
    runs = sorted(glob.glob(f"{WORK}/runs/*_{name}"))
    return runs[-1] if runs else None


def train(config, name, *overrides):
    run = latest_run(name)
    args = [sys.executable, "scripts/train.py", "--config", config, f"max_minutes={MAX_MINUTES}",
            "ckpt_every=1000"]
    args += [f"resume={run}"] if run else [f"name={name}", f"runs_root={WORK}/runs", *overrides]
    print("resuming" if run else "starting", name, run or "")
    subprocess.run(args, check=True)
    return latest_run(name)


def show_curve(run):
    from IPython.display import Image, display
    subprocess.run([sys.executable, "scripts/plot_training.py", "--config",
                    "configs/eval/default.yaml", f"run={run}"], check=True)
    display(Image(f"{run}/training.png"))

In [ ]:
# Unconditional EGNN (main backbone): used by B1, B4 and B5
UNCOND_RUN = train("configs/train/uncond.yaml", "egnn_uncond")
show_curve(UNCOND_RUN)

In [ ]:
# Conditional EGNN (pin mask + defect label + CFG dropout): the main method
COND_RUN = train("configs/train/cond.yaml", "egnn_cond")
show_curve(COND_RUN)

## Experiments

Each experiment samples `N_SAMPLES` structures, relaxes them on the GPU, and evaluates
them against natural defects (95% bootstrap CIs). Finished experiments (a `metrics.json`
in Drive) are skipped, so the cell can be re-run after a disconnect.

Costs scale with `N_SAMPLES × steps`; RePaint with `n_resample = U` costs U times more.
Start with `DEFECTS = ["D2"]` and the default list, then extend.

In [ ]:
UNCOND = f"{latest_run('egnn_uncond')}/ckpt.pt"
COND = f"{latest_run('egnn_cond')}/ckpt.pt"
EVAL = f"{WORK}/eval"

DEFECTS = ["D2", "D1_MINUS", "D1_PLUS"]
N_SAMPLES, BATCH = 256, 64
STEPS = ["schedule.n_noisy=2900", "schedule.n_final=100"]  # DM2 default; try 900/100 first

# (label, sampler config, checkpoint, request mode, extra overrides)
EXPERIMENTS = [
    ("B1_uncond", "dm2", UNCOND, "centre", []),
    ("B4_clamp", "clamp", UNCOND, "patch", []),
    ("B5_repaint", "repaint", UNCOND, "patch", ["strategy.n_resample=1"]),
    ("B5_repaint", "repaint", UNCOND, "host", ["strategy.n_resample=1"]),
    ("cond_w0", "cfg", COND, "centre", ["strategy.guidance_w=0"]),
    ("cond_w2", "cfg", COND, "centre", ["strategy.guidance_w=2"]),
    ("cond_w0", "cfg", COND, "host", ["strategy.guidance_w=0"]),
    ("cond_w2", "cfg", COND, "host", ["strategy.guidance_w=2"]),
]


def done(name):
    return bool(glob.glob(f"{EVAL}/*_{name}/metrics.json"))


def run_script(script, *args):
    subprocess.run([sys.executable, f"scripts/{script}.py", *args], check=True)


def evaluate(run_dir):
    run_script("relax", "--config", "configs/eval/default.yaml", f"run={run_dir}", "batch_size=64")
    run_script("evaluate", "--config", "configs/eval/default.yaml", f"run={run_dir}",
               "data=data/ka2d_256")
    run_script("plot_samples", "--config", "configs/eval/default.yaml", f"run={run_dir}",
               "data=data/ka2d_256", "n=8")


for defect in DEFECTS:
    for label, sampler, ckpt, mode, extra in EXPERIMENTS:
        name = f"{label}_{mode}_{defect}"
        if done(name):
            print("skip (done):", name)
            continue
        for stale in glob.glob(f"{EVAL}/*_{name}"):  # unfinished attempt from a lost session
            subprocess.run(["rm", "-rf", stale])
        run_script("sample", "--config", f"configs/sampler/{sampler}.yaml", f"ckpt={ckpt}",
                   "data=data/ka2d_256", f"defect={defect}", f"request.mode={mode}",
                   f"n_samples={N_SAMPLES}", f"batch_size={BATCH}", f"name={name}",
                   f"runs_root={EVAL}", *STEPS, *extra)
        evaluate(sorted(glob.glob(f"{EVAL}/*_{name}"))[-1])

In [ ]:
# Baseline B2 (hand insertion; no model) and the natural-vs-natural floor, per defect
for defect in DEFECTS:
    name = f"hand_insert_{defect}"
    if not done(name):
        run_script("hand_insert", "--config", "configs/eval/default.yaml", "data=data/ka2d_256",
                   f"defect={defect}", f"n_samples={N_SAMPLES}", f"runs_root={EVAL}")
        run_dir = sorted(glob.glob(f"{EVAL}/*_{name}"))[-1]
        run_script("relax", "--config", "configs/eval/default.yaml", f"run={run_dir}",
                   "batch_size=64")
        run_script("evaluate", "--config", "configs/eval/default.yaml", f"run={run_dir}",
                   "data=data/ka2d_256", "also_unrelaxed=true")
    if not glob.glob(f"{EVAL}/*_floor_{defect}/metrics.json"):
        run_script("evaluate", "--config", "configs/eval/default.yaml", "floor=true",
                   f"defect={defect}", "data=data/ka2d_256", f"runs_root={EVAL}")

### Optional: baseline B6 (local melt-quench)

Needs LAMMPS, which runs on the CPU (Colab has 2 cores: about 2 s per attempt).

In [ ]:
!pip install -q lammps mpich
libmpi = subprocess.run("find / -name libmpi.so.12 -not -path '/proc/*' 2>/dev/null | head -1",
                        shell=True, capture_output=True, text=True).stdout.strip()
os.environ["LD_LIBRARY_PATH"] = os.path.dirname(libmpi) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
for defect in DEFECTS:
    name = f"local_melt_quench_{defect}"
    if done(name):
        continue
    run_script("local_melt_quench", "--config", "configs/data/ka2d_256.yaml", "data=data/ka2d_256",
               f"defect={defect}", f"n_samples={N_SAMPLES}", "workers=2", f"runs_root={EVAL}")
    run_dir = sorted(glob.glob(f"{EVAL}/*_{name}"))[-1]
    run_script("relax", "--config", "configs/eval/default.yaml", f"run={run_dir}", "batch_size=64")
    run_script("evaluate", "--config", "configs/eval/default.yaml", f"run={run_dir}",
               "data=data/ka2d_256")

## Summary

Collects every `metrics.json` into `MyDrive/idl-project/results/summary.csv` (all metrics)
and `summary.md` (headline table), next to the sample figures in each run directory.

In [ ]:
run_script("aggregate_results", "--config", "configs/eval/default.yaml", f"runs_dir={EVAL}",
           f"out={WORK}/results/summary.csv")
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK}/results/summary.md").read()))

## Bringing results back to the repository

Everything stays in Drive. To share results with the team, copy `results/summary.csv`,
`summary.md`, the training curves and a few `relaxed.png` figures into a new folder under
`results/` in the repository (for example `results/2026-10-colab/`) and commit them. Model
checkpoints (`runs/*/ckpt.pt`) are small enough to share from Drive.